# Using [Cortex-JS Compute Engine](https://mathlive.io/compute-engine/) in Raku

Anton Antonov   
March 2026 

---

## Setup

In [1]:
use CortexJS;
use LaTeX::Grammar;

If the CortexJS functions below fail to evaluate execute the following magic cell (uncomment first): 

In [2]:
# #%bash
# npm install @cortex-js/compute-engine

()

---

## Conversion of LaTeX to MathJSON

The LaTeX expressions can be converted to MathJSON using the Cortex-JS functions or by the Raku package ["LaTeX::Grammar"](https://raku.land/zef:antononcube/LaTeX::Grammar). Here is an example with the latter:

In [3]:
latex-interpret('\sqrt{4 * x^2 + 12 * x + 9}')

["Root",["Add",["Add",["Multiply",4,["Power","x",2]],["Multiply",12,"x"]],9],2]

In [4]:
say parse-latex('\\sin(x)').raku;
say parse-latex('\\sin{x}').raku;

$["Sin", "x"]
$["Sin", "x"]


In [5]:
say parse-latex('\\sin^{2}{x}').raku;

$["Power", ["Sin", "x"], 2]


In [6]:
say parse-latex('\sum_{n=1}^{10} n^2').raku;

$["Sum", ["Power", "n", 2], ["Limits", "n", 1, 10]]


In [7]:
say parse-latex('\int_{0}^{1} x^2 dx').raku;

$["Integrate", ["Function", ["Block", ["Power", "x", 2]], "x"], ["Limits", "x", 0, 1]]


In [8]:
say parse-latex('\int_{0}^{1} x^2 + 2 dx').raku;

$["Integrate", ["Function", ["Block", ["Add", ["Power", "x", 2], 2]], "x"], ["Limits", "x", 0, 1]]


In [9]:
say parse-latex('\int 1 dx').raku;

$["Integrate", ["Function", ["Block", 1]], ["Limits", "x", "Nothing", "Nothing"]]


There are some notable difference between the two converters:

In [10]:
#% html
[
    '4 x^2 + 12 x + 9',
    '4 \times x^2 + 12 \times x + 9',
    '4 * x^2 + 12 * x + 9',
    '\sqrt{4x^2 + 12x + 9}',
    '\sqrt{4 * x^2 + 12 * x + 9}',
]
andthen $_.map({ %(expr => "latex«$_»", Cortex-JS => parse-latex($_).raku, 'LaTeX::Grammar' => latex-interpret($_)) })
andthen .&to-html(field-names => <expr Cortex-JS LaTeX::Grammar>, align => 'left')
andthen .subst(/ 'latex«' (.*?) '»' /, { latex-interpret($0.Str, actions => 'MathML')}, :g)

expr,Cortex-JS,LaTeX::Grammar
"""4×x2+12×x+9""","$[""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]","[""Add"",[""Add"",[""Multiply"",4,[""Power"",""x"",2]],[""Multiply"",12,""x""]],9]"
"""4×times×x2+12×times×x+9""","$[""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]","[""Add"",[""Add"",[""Multiply"",[""Multiply"",4,""times""],[""Power"",""x"",2]],[""Multiply"",[""Multiply"",12,""times""],""x""]],9]"
"""4×x2+12×x+9""","$[""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]","[""Add"",[""Add"",[""Multiply"",4,[""Power"",""x"",2]],[""Multiply"",12,""x""]],9]"
"""4×x2+12×x+9""","$[""Sqrt"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]]","[""Root"",[""Add"",[""Add"",[""Multiply"",4,[""Power"",""x"",2]],[""Multiply"",12,""x""]],9],2]"
"""4×x2+12×x+9""","$[""Sqrt"", [""Add"", [""Multiply"", 4, [""Power"", ""x"", 2]], [""Multiply"", 12, ""x""], 9]]","[""Root"",[""Add"",[""Add"",[""Multiply"",4,[""Power"",""x"",2]],[""Multiply"",12,""x""]],9],2]"


----

## Algebraic operations

In [11]:
'e^{i\\pi}'
==> parse-latex()
==> evaluate()

-1

In [12]:
'(a + b)^2'==> 
parse-latex()==> 
expand()==>
to-latex(:!bracketed)

a^2+b^2+2ab

Simplify an expression:

In [13]:
#%markdown
my $expr = parse-latex('3x^2 + 2x^2 + x + 5');
"{to-latex($expr)} = {to-latex(simplify($expr))}";

$2x^2+3x^2+x+5$ = $5x^2+x+5$

Convert to MathJSON first then simplify:

In [14]:
#%markdown
'\sqrt{3x^2 + 2x^2 + x + 5}'
==> parse-latex()
==> simplify()
==> to-latex()

$\sqrt{5x^2+x+5}$

---

## Using sub-wrappers

Instead of having the "long" code:

```raku
'\sqrt{3x^2 + 2x^2 + x + 5}'
==> parse-latex()
==> simplify()
==> to-latex()
```

we can use a wrapper if `simplify` that:

1. Checks if the input is LaTeX spec
2. If LaTeX input spec, converts the input to MathJSON
3. Invokes `simplify`
4. If LaTeX input spec, converts to result to LaTeX

To wrap all symbolic functions / subs of "CortexJS" ise the sub `wrap-symbolic-subs`; to unwrap use `unwrap-symbolic-subs`.

Here the sub wrapping is done:

In [21]:
CortexJS::wrap-symbolic-subs()

[Routine::WrapHandle.new Routine::WrapHandle.new Routine::WrapHandle.new Routine::WrapHandle.new Routine::WrapHandle.new Routine::WrapHandle.new Routine::WrapHandle.new Routine::WrapHandle.new]

Here is the more convenient computation code:

In [16]:
#%markdown
'\sqrt{3x^2 + 2x^2 + x + 5}'
==> CortexJS::simplify()

$\sqrt{5x^2+x+5}$

Another example:

In [22]:
#% markdown
'x^2 - a x + 1 = 0'
==> CortexJS::solve(vars => 'a')

$x+\frac{1}{x}$

Here the sub unwrapping is done:

In [19]:
CortexJS::unwrap-symbolic-subs()

[]

----

## Solve equations

In [23]:
'x^2 - 5 x + 6 = 0'
==> parse-latex() 
==> solve()

[3 2]

In [24]:
#%markdown
'x^2 - 6 x + 1 = 0'
==> parse-latex() 
==> solve()
==> to-latex()

$3+2\sqrt{2}, \: 3-2\sqrt{2}$

In [25]:
#% markdown
'x^2 - a x + 1 = 0'
==> parse-latex() 
==> solve(vars => 'a')
==> to-latex()

$x+\frac{1}{x}$

---

## Calculus

Derivate of expression over the variable $x$:

In [26]:
#%markdown
'D_x (x^2 + 1)'
==> parse-latex()
==> evaluate()
==> to-latex()

$2x$

Evaluate the undefined integral $\int x^2 dx$:

In [27]:
#%markdown
'\int x^2 dx'
==> parse-latex()
==> evaluate()
==> to-latex()

$\frac{x^3}{3}$

Evaluate the defined integral $\int_{0}^{1} x^2 dx$:

In [28]:
#%markdown
'\int_{0}^{1} x^2 dx'
==> parse-latex()
==> evaluate()
==> to-latex()

$\frac{1}{3}$

Evaluate the limit $\lim_{x \to 0} \frac{\sin(x)}{x}$:

In [29]:
'\lim_{x \to 0} \frac{\sin(x)}{x}'
==> parse-latex()
==> evaluate()
==> N()

1

---

## Function definition

See the original JavaScript examples [here](https://mathlive.io/compute-engine/guides/augmenting/).

In [30]:
evaluate(parse-latex('f(x) := 2x'));
evaluate(parse-latex('f(3)'));

6